<a href="https://colab.research.google.com/github/Siva-p-11/Gpu-Computing-Colab/blob/main/Matrix_Addition_3D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Fri Aug 28 08:03:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [18]:
%%writefile matrix_mul_3d.cu

#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define Z 2
#define M 3
#define K 3
#define N 3

__global__ void matrixMul3D(float *A, float *B, float *C)
{
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int z   = blockIdx.z;

    if (row < M && col < N && z < Z)
    {
        float sum = 0.0f;

        for (int k = 0; k < K; k++)
        {
            int indexA = z * M * K + row * K + k;
            int indexB = z * K * N + k * N + col;

            sum += A[indexA] * B[indexB];
        }

        int indexC = z * M * N + row * N + col;

        C[indexC] = sum;
    }
}

int main()
{
    int sizeA = Z * M * K * sizeof(float);
    int sizeB = Z * K * N * sizeof(float);
    int sizeC = Z * M * N * sizeof(float);

    float *h_A = (float*)malloc(sizeA);
    float *h_B = (float*)malloc(sizeB);
    float *h_C = (float*)malloc(sizeC);

    // Initialize matrices
    for (int z = 0; z < Z; z++)
    {
        for (int i = 0; i < M; i++)
        {
            for (int j = 0; j < K; j++)
            {
                int index = z * M * K + i * K + j;
                h_A[index] = i + j + 1;
            }
        }

        for (int i = 0; i < K; i++)
        {
            for (int j = 0; j < N; j++)
            {
                int index = z * K * N + i * N + j;
                h_B[index] = i + j + 1;
            }
        }
    }

    float *d_A, *d_B, *d_C;

    cudaMalloc(&d_A, sizeA);
    cudaMalloc(&d_B, sizeB);
    cudaMalloc(&d_C, sizeC);

    cudaMemcpy(d_A, h_A, sizeA, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, sizeB, cudaMemcpyHostToDevice);

    // 3D grid
    dim3 threadsPerBlock(16, 16, 1);

    dim3 blocksPerGrid(
        (N + 15) / 16,
        (M + 15) / 16,
        Z
    );

    matrixMul3D<<<blocksPerGrid, threadsPerBlock>>>(d_A, d_B, d_C);

    cudaDeviceSynchronize();

    cudaMemcpy(h_C, d_C, sizeC, cudaMemcpyDeviceToHost);

    // Display result
    for (int z = 0; z < Z; z++)
    {
        printf("\n========== Layer Z = %d ==========\n", z);

        printf("\nMatrix A:\n");

        for (int i = 0; i < M; i++)
        {
            for (int j = 0; j < K; j++)
            {
                int index = z * M * K + i * K + j;
                printf("%.0f ", h_A[index]);
            }
            printf("\n");
        }

        printf("\nMatrix B:\n");

        for (int i = 0; i < K; i++)
        {
            for (int j = 0; j < N; j++)
            {
                int index = z * K * N + i * N + j;
                printf("%.0f ", h_B[index]);
            }
            printf("\n");
        }

        printf("\nMatrix C = A x B:\n");

        for (int i = 0; i < M; i++)
        {
            for (int j = 0; j < N; j++)
            {
                int index = z * M * N + i * N + j;
                printf("%.0f ", h_C[index]);
            }
            printf("\n");
        }
    }

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    free(h_A);
    free(h_B);
    free(h_C);

    return 0;
}

Overwriting matrix_mul_3d.cu


In [19]:
!nvcc matrix_mul_3d.cu -o matrix_mul_3d

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [20]:
!./matrix_mul_3d


========== Layer Z = 0 ==========

Matrix A:
1 2 3 
2 3 4 
3 4 5 

Matrix B:
1 2 3 
2 3 4 
3 4 5 

Matrix C = A x B:
14 20 26 
20 29 38 
26 38 50 

========== Layer Z = 1 ==========

Matrix A:
1 2 3 
2 3 4 
3 4 5 

Matrix B:
1 2 3 
2 3 4 
3 4 5 

Matrix C = A x B:
14 20 26 
20 29 38 
26 38 50 
